In [1]:
import polars as pl
import numpy as np
import time
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
import joblib
from tqdm import tqdm
from bertopic import BERTopic
import os

/home/javclamar/Projects/tfg-sentiment-analysis/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
total_rows = 6_990_280
batch_size = 50000
expected_batches = total_rows // batch_size + (1 if total_rows % batch_size != 0 else 0)

def lda_model_training(input_csv):

    # Sample para entrenar LDA, se puede modificar si nos preocupa menos el tiempo
    sample_df = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=300000, seed=42) 
    )
    
    sample_df = sample_df.with_columns(
        pl.col('text').str.replace_all(r'[^a-zA-Z\s]', '').str.to_lowercase().alias('clean_text')
    )

    # Inicializar y entrenar el modelo para vectorizar nuestras reviews
    vectorizer = CountVectorizer(stop_words='english', max_features=25000, max_df=0.85, min_df=10)
    vectorizer.fit(sample_df['clean_text'].to_list())
    
    del sample_df

    # Inicializar el modelo LDA
    lda = LatentDirichletAllocation(
        n_components=15,           
        learning_method='online',  
        n_jobs=-1,                 
        batch_size=50000,          
        random_state=42
    )
    
    start_time = time.time()
    reader = pl.read_csv_batched(input_csv, batch_size=50000)

    # Bucle para entrenar el modelo LDA
    batch_count = 0
    with tqdm(total=expected_batches, desc="Training LDA", unit="batch") as pbar:
        while True:
            batches = reader.next_batches(1)
            if not batches:
                break
            
            chunk = batches[0]
            
            chunk = chunk.with_columns(
                pl.col('text').str.replace_all(r'[^a-zA-Z\s]', '').str.to_lowercase().alias('clean_text')
            )
            
            X_chunk = vectorizer.transform(chunk['clean_text'].to_list())
            
            lda.partial_fit(X_chunk)
            pbar.update(1)
            
    joblib.dump(lda, '../data/models/lda/yelp_lda_model.joblib')
    joblib.dump(vectorizer, '../data/models/lda/yelp_vectorizer.joblib')
    
    print(f"Training completed successfully in {(time.time() - start_time) / 60:.2f} minutes.")

lda_model_training(csv_reviews)

Training LDA:  49%|█████████████████████████████████████████████████▎                                                  | 69/140 [04:19<04:27,  3.76s/batch]

KeyboardInterrupt


KeyboardInterrupt



In [ ]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'

def bertopic_model_training(input_csv):
    os.makedirs('../data/models/bertopic', exist_ok=True)
    
    # Sample para entrenar BERTopic, se puede modificar y aumentar si es necesario
    sample_df = (
        pl.scan_csv(input_csv)
        .select('text')
        .collect()
        .sample(n=200000, seed=42) 
    )
    
    sample_df = sample_df.with_columns(
        pl.col('text').str.replace_all(r'[^a-zA-Z\s]', '').str.to_lowercase().alias('clean_text')
    )
    
    docs = sample_df['clean_text'].to_list()
    del sample_df
    
    start_time = time.time()

    # Entrenamiento de BERTopic
    topic_model = BERTopic(language="english", calculate_probabilities=False, verbose=True)
    topics, probs = topic_model.fit_transform(docs)
    
    topic_model.save('../data/models/bertopic/yelp_bertopic_model', serialization="safetensors", save_ctfidf=True)
    
    print(f"Training completed successfully in {(time.time() - start_time) / 60:.2f} minutes.")

bertopic_model_training(csv_reviews)

In [2]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
csv_reviews_output_topics = '../results/topic_modeling/yelp_academic_dataset_review_topics.csv'
lda_model_path = '../data/models/lda/yelp_lda_model.joblib'
vectorizer_path = '../data/models/lda/yelp_vectorizer.joblib'
total_rows = 6_990_280
batch_size = 50000
expected_batches = total_rows // batch_size + (1 if total_rows % batch_size != 0 else 0)

def topic_modeling(input_csv, output_csv, 
                       lda_model_path='../data/models/lda/yelp_lda_model.joblib', 
                       vectorizer_path='../data/models/lda/yelp_vectorizer.joblib',
                       bertopic_model_path='../data/models/bertopic/yelp_bertopic_model'):

    # Carga los modelos
    lda = joblib.load(lda_model_path)
    vectorizer = joblib.load(vectorizer_path)
    bertopic_model = BERTopic.load(bertopic_model_path)
    bertopic_model.verbose = False

    # Bloque para mostrar las palabras más representativas de los temas en vez de números
    feature_names = vectorizer.get_feature_names_out()
    topic_words_list = []
    for topic in lda.components_:
        top_indices = topic.argsort()[:-4:-1]
        topic_words_list.append(", ".join([feature_names[i] for i in top_indices]))
    topic_words_array = np.array(topic_words_list)
    
    bertopic_labels_dict = {-1: 'outlier'}
    for topic_id in bertopic_model.get_topic_info()['Topic']:
        if topic_id != -1:
            rep = bertopic_model.get_topic(topic_id)
            if rep:
                bertopic_labels_dict[topic_id] = ', '.join([word for word, _ in rep[:3]])
            else:
                bertopic_labels_dict[topic_id] = f'topic_{topic_id}'

    start_time = time.time()

    batch_size = 50000 
    reader = pl.read_csv_batched(input_csv, batch_size=batch_size)

    batch_count = 0

    # Bucle para realizar la inferencia de LDA y BERTopic a la vez
    with tqdm(total=expected_batches, desc="Topic Modeling", unit="batch") as pbar:
        while True:
            batches = reader.next_batches(1)
            if not batches:
                break
            
            chunk = batches[0]
            
            # Limpieza de todo lo que no sea letras y poner en minúsculas
            cleaned_texts = chunk['text'].str.replace_all(r'[^a-zA-Z\s]', '').str.to_lowercase().to_list()
            
            # Inferencia LDA
            X_chunk = vectorizer.transform(cleaned_texts)
            topic_distributions = lda.transform(X_chunk)
            
            dominant_topics_idx = np.argmax(topic_distributions, axis=1)
            dominant_topics = topic_words_array[dominant_topics_idx]
            topic_probs = np.max(topic_distributions, axis=1)
            
            # Inferencia BERTopic
            bertopic_topics, _ = bertopic_model.transform(cleaned_texts)
            bertopic_topic_labels = [bertopic_labels_dict.get(t, f'topic_{t}') for t in bertopic_topics]
            
            # Empaquetar resultados en DataFrame
            chunk = chunk.with_columns([
                    pl.Series('lda_dominant_topic', dominant_topics),
                    pl.Series('lda_topic_probability', topic_probs),
                    pl.Series('bertopic_topic', bertopic_topics),
                    pl.Series('bertopic_dominant_topic', bertopic_topic_labels)
                ])
            
            mode = 'wb' if batch_count == 0 else 'ab'
            with open(output_csv, mode) as f:
                chunk.write_csv(f, include_header=(batch_count == 0))
    
            batch_count += 1
            pbar.update(1)
    
    print(f"Resultados guardados dentro de {output_csv} en {(time.time() - start_time) / 60:.2f} minutes.")

topic_modeling(csv_reviews, csv_reviews_output_topics, lda_model_path, vectorizer_path, bertopic_model_path)

Loading models...


Loading weights: 100%|████████████████████████████████| 103/103 [00:00<00:00, 595.45it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Scoring Reviews: 100%|█████████████████████████████████████████████████████████████████████| 140/140 [1:37:49<00:00, 41.93s/batch]

Saved to ../results/topic_modeling/yelp_academic_dataset_review_topics.csv in 97.83 minutes.


In [3]:
csv_reviews_output_topics = '../results/topic_modeling/yelp_academic_dataset_review_topics.csv'

print(pl.scan_csv(csv_reviews_output_topics, ignore_errors=True)
    .tail(5).collect())

shape: (5, 11)
┌────────────┬────────────┬────────────┬───────┬───┬───────────┬───────────┬───────────┬───────────┐
│ review_id  ┆ user_id    ┆ business_i ┆ stars ┆ … ┆ lda_domin ┆ lda_topic ┆ bertopic_ ┆ bertopic_ │
│ ---        ┆ ---        ┆ d          ┆ ---   ┆   ┆ ant_topic ┆ _probabil ┆ topic     ┆ dominant_ │
│ str        ┆ str        ┆ ---        ┆ i64   ┆   ┆ ---       ┆ ity       ┆ ---       ┆ topic     │
│            ┆            ┆ str        ┆       ┆   ┆ str       ┆ ---       ┆ i64       ┆ ---       │
│            ┆            ┆            ┆       ┆   ┆           ┆ f64       ┆           ┆ str       │
╞════════════╪════════════╪════════════╪═══════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ H0RIamZu0B ┆ qskILQ3k0I ┆ jals67o91g ┆ 5     ┆ … ┆ car,      ┆ 0.813814  ┆ 610       ┆ card,     │
│ 0Ei0P4aeh3 ┆ _qcCMI-k6_ ┆ crD4DC81Vk ┆       ┆   ┆ service,  ┆           ┆           ┆ credit,   │
│ sQ         ┆ QQ         ┆ 6w         ┆       ┆   ┆ told      ┆           ┆